# Lab 1: Clinical Imaging Triage & 55-Language Patient Localization with MedGemma & TranslateGemma

**Enterprise Open-Weight AI Playbook — Part 2 Hands-On Lab (Kaggle / Google Colab / Vertex AI Model Garden)**  
*Authors: Kanchana Patlolla & Anusha Sabbineni · [openweights.cloudrace.info](https://openweights.cloudrace.info/)*

<table><tbody><tr>
  <td style="text-align: center; padding: 8px 16px;">
    <a href="https://colab.research.google.com/github/kanchpat/open-weight-models/blob/main/series/part-02-openmaas-serverless/medgemma_translategemma_clinical_demo.ipynb">
      <img alt="Google Colab" src="https://www.tensorflow.org/images/colab_logo_32px.png" width="32px"><br><b>Run in Google Colab</b>
    </a>
  </td>
  <td style="text-align: center; padding: 8px 16px;">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fkanchpat%2Fopen-weight-models%2Fmain%2Fseries%2Fpart-02-openmaas-serverless%2Fmedgemma_translategemma_clinical_demo.ipynb">
      <img alt="Colab Enterprise" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" width="32px"><br><b>Run in Colab Enterprise</b>
    </a>
  </td>
</tr></tbody></table>

---

### Why Enterprises Combine Domain-Specialized Open-Weight Gemma Models

Small domain-specialized open-weight models can run on infrastructure you control, including a single GPU. This lab chains two of them:

1. **[MedGemma 1.5 4B (`google/medgemma-1.5-4b-it`)](https://developers.google.com/health-ai-developer-foundations/medgemma)**: A Gemma 3 variant with a SigLIP image encoder pre-trained on medical images. Its model card lists support for **2D Chest X-Ray anatomical bounding-box localization**, **longitudinal prior-vs-current scan comparison**, **3D CT/MRI slices**, and **whole-slide histopathology**.
2. **[TranslateGemma 4B (`google/translategemma-4b-it`)](https://huggingface.co/google/translategemma-4b-it)**: A lightweight open-weight translation model for translation across **55 languages** using a structured `{source_lang_code, target_lang_code, text}` chat template.

```
[Chest X-Ray PA / Longitudinal Scans]
          │
          ▼
┌────────────────────────────────────────────────────────┐
│ 1. MedGemma 1.5 4B (google/medgemma-1.5-4b-it)         │
│    • Normalized [0-1000] box_2d Anatomical Localization│
│    • Multi-Image Longitudinal Progression Comparison   │
│    • Clinical Reasoning Trace + Structured Assessment  │
└────────────────────────────────────────────────────────┘
          │
          ▼
┌────────────────────────────────────────────────────────┐
│ 2. TranslateGemma 4B (google/translategemma-4b-it)     │
│    • Patient Care Summary Translation                  │
│    • Español (es) · हिन्दी (hi) · 日本語 (ja) · Français │
└────────────────────────────────────────────────────────┘
```

> **Educational & Research Disclaimer**: This notebook demonstrates baseline capabilities of open-weight Health AI Developer Foundations (HAI-DEF) models. It is not a medical device and should not be used for clinical diagnosis. See the [HAI-DEF Terms of Use](https://developers.google.com/health-ai-developer-foundations/terms).

## Step 1: Environment Setup & Hugging Face Authentication

Run on a GPU runtime (Kaggle: **Accelerator → GPU T4 x2**; Colab: **Runtime → Change runtime type → T4 GPU**), or on a local machine with a GPU.

### Get a Hugging Face token

Both models are gated, so you need a token from an account that has accepted their terms.

1. Create a free account at [huggingface.co/join](https://huggingface.co/join) if you do not have one.
2. While signed in, open each model page and accept the usage conditions:
   - [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)
   - [google/translategemma-4b-it](https://huggingface.co/google/translategemma-4b-it)
3. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens), click **Create new token**, choose the **Read** type, and copy the token (it starts with `hf_`).

### Where to add the token

Do not paste the token into a notebook cell; anyone you share the notebook with would see it.

- **Google Colab:** click the 🔑 **Secrets** tab in the left panel, add a secret named `HF_TOKEN`, paste the token as its value, and turn on **Notebook access**.
- **Kaggle:** open **Add-ons → Secrets**, add a secret with the label `HF_TOKEN`, paste the token, and tick the box to attach it to this notebook.
- **Local machine:** before starting Jupyter, either run `huggingface-cli login` in a terminal and paste the token when asked, or set it as an environment variable with `export HF_TOKEN=hf_...`.

The next cell picks the token up automatically in all three cases.

In [ ]:
# Pillow is deliberately not listed: upgrading it in a running kernel that has
# already imported PIL causes "cannot import name '_Ink' from 'PIL._typing'".
!pip install --quiet "transformers>=4.50.0" accelerate scikit-image gradio

import os
import sys

# Auto-detect Google Colab, Kaggle, or Vertex AI Colab Enterprise
in_colab = "google.colab" in sys.modules and not os.environ.get("VERTEX_PRODUCT")
in_kaggle = os.path.exists("/kaggle/input")

if in_colab:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
elif in_kaggle and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("Kaggle runtime detected. Using attached Kaggle Models or interactive HF login.")

print(f"Runtime ready | Colab={in_colab} | Kaggle={in_kaggle}")

## Step 2: Load `MedGemma 1.5 4B`

MedGemma is loaded on its own, at full `bfloat16` precision (about 9 GB). TranslateGemma is loaded later, in Step 5.

Two 4B models do not fit together on one 15 GB T4, and loading MedGemma in 4-bit to make room gave noticeably worse bounding boxes when we tried it. So on a single GPU the notebook runs the models one after the other. With two GPUs (Kaggle **GPU T4 x2**) both stay loaded.

If you hit an out-of-memory error, restart the runtime so the GPU is empty, then run from the top.

In [ ]:
import gc

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, pipeline

MEDGEMMA_ID = "google/medgemma-1.5-4b-it"
TRANSLATEGEMMA_ID = "google/translategemma-4b-it"

assert torch.cuda.is_available(), "No GPU found. Switch to a GPU runtime and rerun from the top."
TWO_GPUS = torch.cuda.device_count() >= 2


def gpu_report() -> None:
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {(total - free) / 1e9:.1f} GB used of {total / 1e9:.1f} GB")


def strip_thinking(text: str) -> str:
    """Removes MedGemma's optional thinking trace (<unused94>...<unused95>)."""
    return text.split("<unused95>", 1)[1].lstrip() if "<unused95>" in text else text


# Drop models left over from an earlier run before loading again.
for _name in ("medgemma_pipe", "translate_model"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

print(f"Loading {MEDGEMMA_ID} on GPU 0 (bfloat16)...")
# {"": 0} keeps the whole model on GPU 0 and fails loudly instead of silently
# spilling layers onto the CPU, which makes generation extremely slow.
medgemma_pipe = pipeline(
    "image-text-to-text",
    model=MEDGEMMA_ID,
    model_kwargs=dict(torch_dtype=torch.bfloat16, device_map={"": 0}),
)
medgemma_pipe.model.generation_config.do_sample = False
gpu_report()

## Step 3: Anatomical Bounding-Box Localization on Chest X-Ray (`MedGemma`)

`MedGemma 1.5 4B` is trained on the **Chest ImaGenome** anatomical localization dataset to emit normalized `[y0, x0, y1, x1]` bounding boxes on a `[0, 1000]` coordinate grid alongside clinical chain-of-thought reasoning.

Below, we load a public Posteroanterior (PA) Chest X-Ray (`Stillwaterising, CC0, via Wikimedia Commons`), pad it to a square aspect ratio matching model pretraining, and ask `MedGemma` to localize the **right clavicle**.

In [ ]:
import json
import os
import numpy as np
import skimage
from PIL import Image, ImageDraw
from IPython.display import display, Markdown

# 1. Download public CC0 Chest X-Ray sample from Google Health AI Developer Foundations vault
cxr_url = "https://storage.googleapis.com/hai-cd3-foundations-public-vault-entry/med_gemma/colab_example/cxr/chest_xray_pa_3-8-2010.jpeg"
!wget -nc -q {cxr_url}
raw_cxr = Image.open(os.path.basename(cxr_url))

def pad_image_to_square(image_pil):
    image_array = skimage.util.img_as_ubyte(image_pil)
    if len(image_array.shape) < 3:
        image_array = skimage.color.gray2rgb(image_array)
    if image_array.shape[2] == 4:
        image_array = skimage.color.rgba2rgb(image_array)
    h, w = image_array.shape[:2]
    if h < w:
        dh = w - h
        image_array = np.pad(image_array, ((dh // 2, dh - dh // 2), (0, 0), (0, 0)))
    elif w < h:
        dw = h - w
        image_array = np.pad(image_array, ((0, 0), (dw // 2, dw - dw // 2), (0, 0)))
    return Image.fromarray((image_array * 255).astype(np.uint8)) if image_array.max() <= 1.0 else Image.fromarray(image_array)

cxr_square = pad_image_to_square(raw_cxr)

object_name = "right clavicle"  # @param {type: "string"}
localization_prompt = f"""Instructions:
The following user query will require outputting bounding boxes. The format of bounding boxes coordinates is [y0, x0, y1, x1] where (y0, x0) must be top-left corner and (y1, x1) the bottom-right corner. This implies that x0 < x1 and y0 < y1. Always normalize the x and y coordinates the range [0, 1000], meaning that a bounding box starting at 15% of the image width would be associated with an x coordinate of 150. You MUST output a single parseable json list of objects enclosed into ```json...``` brackets, for instance ```json[{{"box_2d": [800, 3, 840, 471], "label": "car"}}, {{"box_2d": [400, 22, 600, 73], "label": "dog"}}]``` is a valid output. Now answer to the user query.

Remember "left" refers to the patient's left side where the heart is and sometimes underneath an L in the upper right corner of the image.

Query:
Where is the {object_name}? Don't give a final answer without reasoning. Output the final answer in the format "Final Answer: X" where X is a JSON list of objects. The object needs a "box_2d" and "label" key. Answer:"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": cxr_square},
            {"type": "text", "text": localization_prompt},
        ],
    }
]

output = medgemma_pipe(text=messages, max_new_tokens=1000, do_sample=False)
response = output[0]["generated_text"][-1]["content"]
if "<unused95>" in response:
    thinking_trace, response = response.split("<unused95>", 1)
    response = response.lstrip()

# Parse normalized [0, 1000] bounding boxes and overlay onto 512x512 X-Ray
json_start = response.find("```json")
json_end = response.rfind("```")
bbox_data = json.loads(response[json_start + 7 : json_end].strip()) if json_start != -1 else []

annotated_cxr = cxr_square.resize((512, 512)).copy()
draw = ImageDraw.Draw(annotated_cxr)
for item in bbox_data:
    y0, x0, y1, x1 = item["box_2d"]
    px0, py0, px1, py1 = x0 / 1000 * 512, y0 / 1000 * 512, x1 / 1000 * 512, y1 / 1000 * 512
    draw.rectangle([(px0, py0), (px1, py1)], outline="#ef4444", width=3)
    draw.text((px0, max(0, py0 - 16)), item.get("label", object_name), fill="#ef4444")

display(Markdown(f"**[ MedGemma Localization Output ]**\n\n{response}"))
display(annotated_cxr)

## Step 4: Longitudinal Chest X-Ray Progression Analysis (`PRIOR` vs. `FOLLOW-UP`)

In clinical practice, radiologists rarely read a Chest X-Ray in isolation — they compare **current scans against historical baseline scans** to track disease progression or treatment response. `MedGemma 1.5 4B` natively accepts multiple radiographs in a single prompt to perform longitudinal comparison.

In [ ]:
# Download Prior (Baseline) and Follow-Up Chest X-Rays
prior_url = "https://storage.googleapis.com/hai-cd3-foundations-public-vault-entry/med_gemma/colab_example/cxr/longitudinal_cxr_before.png"
followup_url = "https://storage.googleapis.com/hai-cd3-foundations-public-vault-entry/med_gemma/colab_example/cxr/longitudinal_cxr_after.png"
!wget -nc -q {prior_url} {followup_url}

cxr_prior = Image.open(os.path.basename(prior_url))
cxr_followup = Image.open(os.path.basename(followup_url))

longitudinal_prompt = """Provide a comparison of these two images and include details from
the image which clinicians and students should take note of when reading longitudinal CXR."""

longitudinal_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": cxr_prior},
            {"type": "image", "image": cxr_followup},
            {"type": "text", "text": longitudinal_prompt},
        ],
    }
]

longitudinal_out = medgemma_pipe(text=longitudinal_messages, max_new_tokens=700)
longitudinal_report = longitudinal_out[0]["generated_text"][-1]["content"]
if "<unused95>" in longitudinal_report:
    longitudinal_report = longitudinal_report.split("<unused95>", 1)[1].lstrip()

display(Markdown("### Longitudinal Scan Pair: Baseline (Left) vs. Follow-Up (Right)"))
display(cxr_prior.resize((300, 300)))
display(cxr_followup.resize((300, 300)))
display(Markdown(f"---\n\n**[ MedGemma Longitudinal Assessment ]**\n\n{longitudinal_report}\n\n---"))

## Step 5: Patient Summary and Translation (`MedGemma` → `TranslateGemma 4B`)

This step chains the two models:

1. **MedGemma** rewrites its own Step 4 comparison as a short plain-language summary.
2. On a single GPU, MedGemma is then unloaded to make room.
3. **TranslateGemma 4B** (`google/translategemma-4b-it`) translates that summary into **Spanish (`es`)**, **Hindi (`hi`)** and **Japanese (`ja`)** using its `{source_lang_code, target_lang_code, text}` chat schema.

Both the summary and the translations are model output. Have a qualified reviewer check them before any real-world use.

In [ ]:
# 1. MedGemma turns its own Step 4 report into a plain-language summary.
if "medgemma_pipe" in globals():
    summary_prompt = (
        "Below is a radiology comparison of two chest X-rays. Rewrite it as a plain-language "
        "summary for the patient in at most three sentences. Use only what the report says; "
        "do not add findings, diagnoses or advice.\n\nReport:\n" + longitudinal_report
    )
    summary_out = medgemma_pipe(
        text=[{"role": "user", "content": [{"type": "text", "text": summary_prompt}]}],
        max_new_tokens=300,
    )
    patient_summary_en = strip_thinking(summary_out[0]["generated_text"][-1]["content"]).strip()
elif "patient_summary_en" not in globals():
    raise RuntimeError("MedGemma is not loaded. Rerun from Step 2.")
print(f"[English summary from MedGemma]: {patient_summary_en}\n")

# 2. On a single GPU, unload MedGemma so TranslateGemma fits at full precision.
if not TWO_GPUS and "medgemma_pipe" in globals():
    del medgemma_pipe
    gc.collect()
    torch.cuda.empty_cache()

# 3. Load TranslateGemma (once).
if "translate_model" not in globals():
    translate_gpu = 1 if TWO_GPUS else 0
    print(f"Loading {TRANSLATEGEMMA_ID} on GPU {translate_gpu} (bfloat16)...")
    translate_processor = AutoProcessor.from_pretrained(TRANSLATEGEMMA_ID)
    translate_model = AutoModelForImageTextToText.from_pretrained(
        TRANSLATEGEMMA_ID, torch_dtype=torch.bfloat16, device_map={"": translate_gpu}
    )
    gpu_report()


def translate_clinical_summary(text: str, source_lang: str, target_lang: str) -> str:
    """Translates text using TranslateGemma's structured chat template."""
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "source_lang_code": source_lang,
                    "target_lang_code": target_lang,
                    "text": text,
                }
            ],
        }
    ]
    inputs = translate_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(translate_model.device)
    with torch.inference_mode():
        outputs = translate_model.generate(**inputs, max_new_tokens=400, do_sample=False)
    return translate_processor.decode(
        outputs[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True
    ).strip()


target_languages = [
    ("es", "Español (Spanish)"),
    ("hi", "हिन्दी (Hindi)"),
    ("ja", "日本語 (Japanese)"),
]

translations = {}
for code, label in target_languages:
    translations[code] = translate_clinical_summary(patient_summary_en, "en", code)
    print(f"[{label} ({code})]: {translations[code]}\n")

## Step 6 (optional, two GPUs only): Interactive Gradio Studio

The UI needs MedGemma and TranslateGemma loaded at the same time, so it only works with two GPUs (Kaggle **GPU T4 x2**). On a single GPU the cell below prints a note and does nothing.

With two GPUs, uncomment `demo.launch(share=True)` at the end of the cell: upload a chest X-ray, localize a landmark with MedGemma, and translate the patient summary with TranslateGemma. This cell has not been tested by the authors.

In [ ]:
import gradio as gr

LANGUAGE_CHOICES = [
    ("Español (Spanish)", "es"),
    ("हिन्दी (Hindi)", "hi"),
    ("日本語 (Japanese)", "ja"),
    ("Français (French)", "fr"),
    ("Deutsch (German)", "de-DE"),
    ("Português (Portuguese)", "pt"),
]

def run_triage_and_translate(image_pil, anatomical_target, target_lang_code):
    sq_img = pad_image_to_square(image_pil)
    prompt = (
        "Instructions: Output bounding boxes as [y0, x0, y1, x1] normalized to [0, 1000] "
        "in a ```json...``` list of objects with 'box_2d' and 'label' keys.\n"
        f"Query: Where is the {anatomical_target}? Provide brief clinical reasoning first, "
        "then Final Answer: ```json[...]```."
    )
    msgs = [{"role": "user", "content": [{"type": "image", "image": sq_img}, {"type": "text", "text": prompt}]}]
    out = medgemma_pipe(text=msgs, max_new_tokens=600, do_sample=False)
    resp = out[0]["generated_text"][-1]["content"]
    if "<unused95>" in resp:
        resp = resp.split("<unused95>", 1)[1].lstrip()

    # Draw bounding box
    vis_img = sq_img.resize((512, 512)).copy()
    j_start, j_end = resp.find("```json"), resp.rfind("```")
    if j_start != -1 and j_end != -1:
        boxes = json.loads(resp[j_start + 7 : j_end].strip())
        d = ImageDraw.Draw(vis_img)
        for b in boxes:
            y0, x0, y1, x1 = b["box_2d"]
            d.rectangle([(x0 / 1000 * 512, y0 / 1000 * 512), (x1 / 1000 * 512, y1 / 1000 * 512)], outline="#ef4444", width=3)

    localized_summary = translate_clinical_summary(patient_summary_en, "en", target_lang_code)
    return vis_img, resp, localized_summary

BOTH_LOADED = "medgemma_pipe" in globals() and "translate_model" in globals()
if not BOTH_LOADED:
    print("Skipped: this UI needs both models loaded, which requires two GPUs (Kaggle GPU T4 x2).")
else:
    with gr.Blocks(theme=gr.themes.Soft(), title="MedGemma + TranslateGemma Clinical Studio") as demo:
        gr.Markdown("## 🩺 MedGemma 1.5 4B + TranslateGemma 4B — Multimodal Clinical & 55-Language Studio")
        with gr.Row():
            with gr.Column():
                img_in = gr.Image(type="pil", value=cxr_square, label="Input Chest X-Ray (PA View)")
                target_in = gr.Textbox(value="right clavicle", label="Anatomical Structure / Finding to Localize")
                lang_in = gr.Dropdown(choices=LANGUAGE_CHOICES, value="es", label="TranslateGemma Target Language (55 Languages)")
                run_btn = gr.Button("Run MedGemma Localization + TranslateGemma Summary", variant="primary")
            with gr.Column():
                img_out = gr.Image(type="pil", label="MedGemma Normalized [0-1000] Bounding-Box Overlay")
                med_out = gr.Textbox(lines=4, label="MedGemma 1.5 4B Clinical Output & Bounding-Box JSON")
                trans_out = gr.Textbox(lines=4, label="TranslateGemma 4B Localized Patient Care Summary")
        run_btn.click(run_triage_and_translate, inputs=[img_in, target_in, lang_in], outputs=[img_out, med_out, trans_out])

    # Uncomment to launch live interactive Gradio server in Kaggle / Colab:
    # demo.launch(share=True)